In [1]:
print(123)

123


In [2]:
q1 = "I just discovered the course, can I still join?"
q2 = "I just just found out about the program, can I still enroll?"

#discover, course, join
#find, program, enroll

MODEL_ID="text-embedding-embeddinggemma-300m-qat"

In [3]:
from dotenv import load_dotenv
from lmstudio import get_lmstudio_client

load_dotenv()

client = get_lmstudio_client()

In [4]:
response = client.embeddings.create(
    input=[q1, q2],
    model=MODEL_ID,  # your loaded embedding model id
)

In [5]:
vectors = [item.embedding for item in response.data]
for i in vectors:
    print(len(i), i[:5])


768 [0.08023923635482788, 0.0258136298507452, 0.013815341517329216, -0.0008619153522886336, 0.05170628800988197]
768 [0.08092005550861359, 0.03175115957856178, -0.0051413727924227715, 0.005817008204758167, 0.03272236883640289]


In [6]:
import numpy as np
v1 = np.array(vectors[0])
v2 = np.array(vectors[1])
cosine_similarity = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
cosine_similarity


np.float64(0.7779481515809346)

In [7]:
q3 = "the weather today is not good"

In [8]:
vector3 = client.embeddings.create(
    input=[q3],
    model=MODEL_ID,  # your loaded embedding model id
).data[0].embedding

v3 = np.array(vector3)

cosine_similarity = np.dot(v1, v3) / (np.linalg.norm(v1) * np.linalg.norm(v3))
cosine_similarity


np.float64(0.340506634515514)

### Fetch the documents

In [9]:
from ingest import load_faq_data
documents = load_faq_data()

In [10]:
documents[10]

{'id': '316180784f',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'}

In [11]:
texts =[]
for doc in documents:
    text = doc['question'] + ' ' + doc['answer']
    texts.append(text)

### Embed the documents
len(texts)



1350

In [12]:
from tqdm.auto import tqdm

batch_size = 50

vectors = []
for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i+batch_size]
    batch_vectors = client.embeddings.create(input=batch, model=MODEL_ID)
    vectors.extend([item.embedding for item in batch_vectors.data])

  0%|          | 0/27 [00:00<?, ?it/s]

In [13]:
len(vectors)

1350

In [14]:
np.array(vectors[0]).shape

(768,)

In [15]:
np_vectors = [np.array(v) for v in vectors]

In [16]:
np_vectors[0].shape

(768,)

In [17]:
scores = []
for i in range(len(np_vectors)):
    score = v1.dot(np_vectors[i])
    scores.append(score)

In [18]:
scores

[np.float64(0.5636069375009292),
 np.float64(0.43939997060466685),
 np.float64(0.6938227747982815),
 np.float64(0.5824668085961088),
 np.float64(0.45545679796307825),
 np.float64(0.47166745626763307),
 np.float64(0.4016405948792213),
 np.float64(0.59227601053056),
 np.float64(0.5117693526137241),
 np.float64(0.4214559786497732),
 np.float64(0.4696461097682671),
 np.float64(0.4832302966264178),
 np.float64(0.4592335034027264),
 np.float64(0.5053637475757827),
 np.float64(0.5006303725488626),
 np.float64(0.4024969638794478),
 np.float64(0.47416025548086493),
 np.float64(0.3359228199167291),
 np.float64(0.44678502366211137),
 np.float64(0.4054207092340203),
 np.float64(0.4950459436428729),
 np.float64(0.40908658154253064),
 np.float64(0.34513728388287357),
 np.float64(0.47796923245773254),
 np.float64(0.4825188334321805),
 np.float64(0.40142073402274114),
 np.float64(0.45324317896969213),
 np.float64(0.32436574779133165),
 np.float64(0.4032409650205282),
 np.float64(0.26683533363328005),


In [19]:
X = np.array(np_vectors)

In [20]:
scores = X.dot(v1)
scores


array([0.56360694, 0.43939997, 0.69382277, ..., 0.32029797, 0.38492066,
       0.34292019], shape=(1350,))

In [21]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(538), np.float64(0.7618314992709485))

In [22]:
documents[538]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [23]:
top5 = np.argsort(scores)[-5:]
top5 = top5[::-1]

In [24]:
documents[907]

{'id': '41aabbd7c5',
 'course': 'machine-learning-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'The course has already started. Can I still join it?',
 'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'}

In [25]:
for i in top5:
    print(documents[i])
    print('-'*100)


{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}
----------------------------------------------------------------------------------------------------
{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."}
----------------------------------------------------------------------------------------------------
{'id': '3f1424af17', 'course': 'data-engineering-zoom

In [26]:
top5 = np.argsort(-scores)[:5]

In [27]:
top5

array([538, 625,   2, 907, 503])